# <font color="steelblue">26. Distribución muestral y error estadístico (simulación de Monte Carlo)</font>

*Unidad 5. Conceptos básicos de inferencia estadística* · Sesión teórica 26 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`salarios_redaccion.csv`, `verificaciones.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["salarios_redaccion.csv", "verificaciones.csv"]
PREGUNTAS = ["q_P26_1.json", "q_P26_2.json", "q_P26_3.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Si cinco institutos hacen la misma encuesta el mismo día, darán cinco resultados distintos aunque todos trabajen bien. Esa variación es el **error muestral**, y se puede medir. En este cuaderno lo veremos con una **simulación de Monte Carlo**: repetir miles de veces una encuesta en el ordenador y mirar cómo se reparten sus resultados.

**Qué necesitas saber antes**: los cuadernos 24 y 25.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Simular la distribución muestral de una proporción y de una media.
2. Calcular e interpretar el error típico.
3. Ver cómo disminuye el error al aumentar la muestra.
4. Entender el teorema central del límite.

## <font color='steelblue'>Contenidos</font>

1. Muchas encuestas imaginarias
2. El error típico
3. El teorema central del límite

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Muchas encuestas imaginarias</font>

**Qué vamos a hacer**: imaginar una población en la que el **34 %** votaría al partido A. Hacemos 10.000 encuestas de 1.000 personas y guardamos el porcentaje de A en cada una. Ese conjunto de resultados es la **distribución muestral** de la proporción.

In [ ]:
rng = np.random.default_rng(26)
p_real = 0.34
n = 1000
resultados = rng.binomial(n, p_real, size=10_000) / n

plt.hist(resultados * 100, bins=40, color="steelblue", edgecolor="white")
plt.axvline(p_real * 100, color="firebrick", label="Valor real: 34 %")
plt.xlabel("% de voto a A en cada encuesta")
plt.ylabel("Número de encuestas")
plt.title("10.000 encuestas de 1.000 personas (simulación)")
plt.legend()
plt.show()
print(f"Media de las encuestas: {resultados.mean():.4f}   Desviación típica: {resultados.std():.4f}")

**Cómo leerlo**: las encuestas se reparten en forma de campana alrededor del 34 %. La mayoría cae entre el 31 % y el 37 %; muy pocas se alejan más. La desviación típica de esta distribución (unos 1,5 puntos) se llama **error típico**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P26_1.json")

## <font color='steelblue'>2. El error típico</font>

El **error típico** de una proporción se calcula sin simular:

$$EE(\hat{p}) = \sqrt{\frac{p(1-p)}{n}}$$

y el de una media, $EE(\bar{x}) = \sigma / \sqrt{n}$. Fíjate en la $\sqrt{n}$: para **reducir el error a la mitad hay que cuadruplicar la muestra**.

In [ ]:
print(f"Error típico teórico: {np.sqrt(p_real * (1 - p_real) / n):.4f}   simulado: {resultados.std():.4f}")
for n_ in [250, 1000, 4000]:
    print(f"n = {n_:>5}: error típico = {np.sqrt(p_real * (1 - p_real) / n_) * 100:.2f} puntos")

In [ ]:
for n_, color in [(250, "lightgrey"), (1000, "steelblue"), (4000, "navy")]:
    r = rng.binomial(n_, p_real, size=10_000) / n_ * 100
    plt.hist(r, bins=50, alpha=0.6, color=color, label=f"n = {n_}", density=True)
plt.axvline(34, color="firebrick")
plt.legend()
plt.xlabel("% de voto a A")
plt.title("Cuanto mayor la muestra, más estrecha la campana")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P26_2.json")

## <font color='steelblue'>3. El teorema central del límite</font>

¿Y si la variable no es normal, como los salarios? El **teorema central del límite** dice que, con muestras suficientemente grandes, **la media de la muestra se distribuye aproximadamente como una normal**, sea cual sea la forma de la variable. Lo comprobamos con los salarios (muy asimétricos):

In [ ]:
sal = pd.read_csv("salarios_redaccion.csv")["salario_bruto_anual"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(sal, bins=30, color="grey")
axes[0].set_title("Salarios (asimétricos)")
for ax, n_ in zip(axes[1:], [5, 50]):
    medias = [sal.sample(n_, replace=True, random_state=i).mean() for i in range(3000)]
    ax.hist(medias, bins=40, color="steelblue")
    ax.set_title(f"Medias de muestras de {n_}")
plt.tight_layout()
plt.show()

**Qué ha pasado**: con muestras de 5 la distribución de la media aún arrastra algo de asimetría; con muestras de 50 ya es una campana. Por eso podemos usar la normal para calcular el error de medias y proporciones aunque los datos no sean normales.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P26_3.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Simula 10.000 encuestas de 400 personas con un 50 % real de «sí». ¿Qué proporción de encuestas da un resultado por debajo del 45 %?

<details><summary>Ver solución</summary>

```python
r = rng.binomial(400, 0.5, size=10_000) / 400
(r < 0.45).mean()
```
</details>

### **Ejercicio 2** *(sección 2)*

Calcula el error típico (en puntos) de una encuesta de 1.500 personas para un partido con un 10 % de voto y para otro con un 50 %. ¿Cuál tiene más error?

<details><summary>Ver solución</summary>

```python
for p_ in [0.1, 0.5]:
    print(p_, np.sqrt(p_ * (1 - p_) / 1500) * 100)
```

El del 50 %: el error es máximo cuando p = 0,5.
</details>

### **Ejercicio 3** *(sección 2)*

¿Cuántas entrevistas hacen falta para que el error típico de una proporción del 50 % sea de 1 punto? Prueba valores de n.

<details><summary>Ver solución</summary>

```python
for n_ in [1000, 2000, 2500, 3000]:
    print(n_, np.sqrt(0.25 / n_) * 100)
```
</details>

### **Ejercicio 4** *(sección 3)*

Repite la simulación del teorema central del límite con la variable «veces compartido» de `verificaciones.csv`, aún más asimétrica, con muestras de 5, 50 y 500. ¿Con qué tamaño ves una campana?

<details><summary>Ver solución</summary>

```python
vc = pd.read_csv("verificaciones.csv")["veces_compartido"]
for n_ in [5, 50, 500]:
    medias = [vc.sample(n_, replace=True, random_state=i).mean() for i in range(2000)]
    plt.hist(medias, bins=40)
    plt.title(f"n = {n_}")
    plt.show()
```

Con asimetrías extremas hace falta una muestra bastante mayor que con los salarios.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- La **distribución muestral** describe cómo varía un estadístico de una muestra a otra.
- Su desviación típica es el **error típico**: $\sqrt{p(1-p)/n}$ para proporciones y $\sigma/\sqrt{n}$ para medias.
- El error disminuye con $\sqrt{n}$: **cuadruplicar la muestra reduce el error a la mitad**.
- **Teorema central del límite**: con muestras grandes, la media muestral es aproximadamente normal.
- La **simulación de Monte Carlo** permite ver todo esto repitiendo miles de muestras imaginarias.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Distribución muestral** | Distribución de un estadístico en todas las muestras posibles. |
| **Error típico** | Desviación típica de la distribución muestral. |
| **Error muestral** | Diferencia entre el estadístico y el parámetro debida al azar del muestreo. |
| **Teorema central del límite** | Con muestras grandes, la media muestral se distribuye aproximadamente como una normal. |
| **Monte Carlo** | Método que usa simulaciones aleatorias para estudiar un problema. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 27 convertirás el error típico en el margen de error de una encuesta.

**Siguiente cuaderno**: [27. Intervalo de confianza para una proporción: el margen de error](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P27_margen_de_error_de_una_encuesta.ipynb)